# Specific Test 2.5 - LM-JEPA for Squared Amplitude Calculation

Two-phase training:
1. **JEPA pretraining** : self-supervised, span masking on all amplitude data
2. **Seq2seq finetuning** : encoder from JEPA + fresh decoder, trained per physics model (QED / QCD)

Baseline: same architecture without JEPA pretraining.

In [ ]:
# clone repo and install deps
!git clone https://github.com/invi-bhagyesh/jepa.git 2>/dev/null || echo "already cloned"
%cd jepa
!pip install -q -r requirements.txt

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from src.data.tokenizer import PhysicsTokenizer
from src.data.dataset import get_loader
from src.models.jepa import JEPA
from src.models.seq2seq import Seq2Seq
from src.training.pretrain import pretrain_jepa
from src.training.finetune import finetune_seq2seq
from src.training.evaluate import evaluate_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# load preprocessed data from notebook 01
tokenizer = PhysicsTokenizer.load("data/processed/vocab.txt")
train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/val.csv")
test_df = pd.read_csv("data/processed/test.csv")

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

D_MODEL = 256
NHEAD = 4
NUM_LAYERS = 4
DIM_FF = 1024
DROPOUT = 0.1
BATCH_SIZE = 64

## 1. JEPA Pretraining

Self-supervised pretraining on all amplitude sequences (QED + QCD combined). The context encoder learns to predict masked span representations in latent space, using the EMA target encoder as the prediction target.

In [ ]:
import os
from pathlib import Path

Path("checkpoints").mkdir(exist_ok=True)

pretrain_loader = get_loader(train_df, tokenizer, batch_size=BATCH_SIZE, shuffle=True)

jepa_model = JEPA(
    vocab_size=tokenizer.vocab_size,
    d_model=D_MODEL, nhead=NHEAD, num_layers=NUM_LAYERS,
    dim_ff=DIM_FF, dropout=DROPOUT
)

n_params = sum(p.numel() for p in jepa_model.parameters() if p.requires_grad)
print(f"JEPA trainable parameters: {n_params / 1e6:.2f}M")

JEPA_CKPT = "checkpoints/jepa_pretrained.pt"
if os.path.exists(JEPA_CKPT):
    print(f"Loading pretrained JEPA from {JEPA_CKPT}")
    jepa_model.load_state_dict(torch.load(JEPA_CKPT, map_location=device))
    jepa_model.to(device)
    jepa_history = {"loss": []}
else:
    jepa_history = pretrain_jepa(
        jepa_model, pretrain_loader,
        epochs=50, lr=1e-4, mask_ratio=0.35,
        device=device
    )
    torch.save(jepa_model.state_dict(), JEPA_CKPT)
    print(f"Saved JEPA checkpoint to {JEPA_CKPT}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(jepa_history["loss"])
plt.xlabel("Epoch")
plt.ylabel("Smooth L1 Loss")
plt.title("JEPA Pretraining Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Seq2Seq Finetuning (QED and QCD separately)

Initialize the encoder from the pretrained JEPA context encoder, attach a fresh decoder, and finetune with teacher-forced cross-entropy on amplitude → squared amplitude.

In [ ]:
results = {}

for model_type in ["QED", "QCD"]:
    print(f"\n{'='*60}")
    print(f"  Training JEPA-pretrained model for {model_type}")
    print(f"{'='*60}")

    ckpt_path = f"checkpoints/jepa_seq2seq_{model_type}.pt"

    tr = train_df[train_df["physics_model"] == model_type]
    va = val_df[val_df["physics_model"] == model_type]
    te = test_df[test_df["physics_model"] == model_type]

    train_loader = get_loader(tr, tokenizer, batch_size=BATCH_SIZE)
    val_loader = get_loader(va, tokenizer, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = get_loader(te, tokenizer, batch_size=BATCH_SIZE, shuffle=False)

    model = Seq2Seq.from_pretrained_encoder(
        jepa_model, tgt_vocab_size=tokenizer.vocab_size,
        d_model=D_MODEL, nhead=NHEAD, num_layers=NUM_LAYERS,
        dim_ff=DIM_FF, dropout=DROPOUT
    )

    if os.path.exists(ckpt_path):
        print(f"Loading finetuned model from {ckpt_path}")
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        model.to(device)
        history = {"train_loss": [], "val_loss": []}
    else:
        history = finetune_seq2seq(
            model, train_loader, val_loader,
            epochs=100, lr=3e-4, patience=10, device=device
        )
        torch.save(model.state_dict(), ckpt_path)
        print(f"Saved to {ckpt_path}")

    metrics = evaluate_model(model, test_loader, tokenizer, device=device)
    results[f"JEPA_{model_type}"] = metrics
    print(f"\n{model_type} results: {metrics}")

    if history["train_loss"]:
        plt.figure(figsize=(8, 4))
        plt.plot(history["train_loss"], label="Train")
        plt.plot(history["val_loss"], label="Val")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title(f"JEPA-pretrained {model_type} -- Finetuning Loss")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

## 3. Baseline (no JEPA pretraining)

Same architecture and hyperparameters, trained from scratch. This isolates the effect of JEPA pretraining.

In [ ]:
for model_type in ["QED", "QCD"]:
    print(f"\n{'='*60}")
    print(f"  Training BASELINE (no JEPA) for {model_type}")
    print(f"{'='*60}")

    ckpt_path = f"checkpoints/baseline_{model_type}.pt"

    tr = train_df[train_df["physics_model"] == model_type]
    va = val_df[val_df["physics_model"] == model_type]
    te = test_df[test_df["physics_model"] == model_type]

    train_loader = get_loader(tr, tokenizer, batch_size=BATCH_SIZE)
    val_loader = get_loader(va, tokenizer, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = get_loader(te, tokenizer, batch_size=BATCH_SIZE, shuffle=False)

    baseline = Seq2Seq(
        src_vocab_size=tokenizer.vocab_size,
        tgt_vocab_size=tokenizer.vocab_size,
        d_model=D_MODEL, nhead=NHEAD, num_layers=NUM_LAYERS,
        dim_ff=DIM_FF, dropout=DROPOUT
    )

    if os.path.exists(ckpt_path):
        print(f"Loading baseline from {ckpt_path}")
        baseline.load_state_dict(torch.load(ckpt_path, map_location=device))
        baseline.to(device)
        history = {"train_loss": [], "val_loss": []}
    else:
        history = finetune_seq2seq(
            baseline, train_loader, val_loader,
            epochs=100, lr=3e-4, patience=10, device=device
        )
        torch.save(baseline.state_dict(), ckpt_path)
        print(f"Saved to {ckpt_path}")

    metrics = evaluate_model(baseline, test_loader, tokenizer, device=device)
    results[f"Baseline_{model_type}"] = metrics
    print(f"\n{model_type} baseline results: {metrics}")

## 4. Results Comparison

In [ ]:
results_df = pd.DataFrame(results).T
results_df.index.name = "Model"
results_df.columns = ["Token Accuracy", "Sequence Exact Match", "Top-5 Accuracy"]
print(results_df.to_string(float_format=lambda x: f"{x:.4f}"))

## 5. Ablation : Masking Ratio

Compare JEPA pretraining with different masking ratios (20%, 35%, 50%) on QED to see which works best.

In [ ]:
ablation = {}

qed_train = train_df[train_df["physics_model"] == "QED"]
qed_val = val_df[val_df["physics_model"] == "QED"]
qed_test = test_df[test_df["physics_model"] == "QED"]

train_loader_all = get_loader(train_df, tokenizer, batch_size=BATCH_SIZE)
qed_train_loader = get_loader(qed_train, tokenizer, batch_size=BATCH_SIZE)
qed_val_loader = get_loader(qed_val, tokenizer, batch_size=BATCH_SIZE, shuffle=False)
qed_test_loader = get_loader(qed_test, tokenizer, batch_size=BATCH_SIZE, shuffle=False)

for ratio in [0.20, 0.35, 0.50]:
    print(f"\n--- Masking ratio: {ratio:.0%} ---")

    jepa = JEPA(
        vocab_size=tokenizer.vocab_size,
        d_model=D_MODEL, nhead=NHEAD, num_layers=NUM_LAYERS,
        dim_ff=DIM_FF, dropout=DROPOUT
    )
    pretrain_jepa(jepa, train_loader_all, epochs=30, lr=1e-4,
                  mask_ratio=ratio, device=device)

    model = Seq2Seq.from_pretrained_encoder(
        jepa, tgt_vocab_size=tokenizer.vocab_size,
        d_model=D_MODEL, nhead=NHEAD, num_layers=NUM_LAYERS,
        dim_ff=DIM_FF, dropout=DROPOUT
    )
    finetune_seq2seq(model, qed_train_loader, qed_val_loader,
                     epochs=50, lr=3e-4, patience=8, device=device)

    metrics = evaluate_model(model, qed_test_loader, tokenizer, device=device)
    ablation[f"{ratio:.0%}"] = metrics
    print(f"  Results: {metrics}")

ablation_df = pd.DataFrame(ablation).T
print("\nAblation results (QED):")
print(ablation_df.to_string(float_format=lambda x: f"{x:.4f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ablation_df.plot(kind="bar", ax=ax, rot=0)
ax.set_xlabel("Masking Ratio")
ax.set_ylabel("Score")
ax.set_title("Ablation: JEPA Masking Ratio (QED)")
ax.legend(loc="lower right")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()